# Forecast-Arena demo

Head-to-head demo of the three forecasting architectures currently implemented:

1. **Single LLM** — one model, one call. The baseline.
2. **Voting Swarm** — N models answer independently; final prediction is a weighted average.
3. **LLM Council** — swarm + a peer-critique round; aggregation weights each agent by the mean reasoning-quality score it receives from peers.

All three share the same output schema (`prediction`, `confidence`, `reasoning`, plus rolled-up tokens/cost/time), so they're directly comparable.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "src"))

from dotenv import load_dotenv
load_dotenv()

from IPython.display import HTML, Markdown, display

from forecast_arena import Agent
from forecast_arena.configs.single import SingleLLM
from forecast_arena.configs.swarm import VotingSwarm
from forecast_arena.configs.council import LLMCouncil


DEFAULT_MODELS = {
    "anthropic": "claude-haiku-4-5",
    "openai": "gpt-4.1-mini",
    "google": "gemini-2.5-flash",
    "moonshot": "kimi-k2.6",
    "deepseek": "deepseek-chat",
    "alibaba": "qwen-plus",
    "meta": "meta-llama/Meta-Llama-3.1-8B-Instruct-Turbo",
}


def make_agents(providers):
    return [Agent(p, DEFAULT_MODELS[p]) for p in providers]


def show(result, *, title):
    f = result.forecast
    display(Markdown(
        f"### {title}\n"
        f"- **Prediction:** {f.prediction:.2f}\n"
        f"- **Confidence:** {f.confidence:.2f}\n"
        f"- **Cost:** \${result.total_cost_usd:.6f}\n"
        f"- **Tokens:** {result.total_input_tokens} in / {result.total_output_tokens} out\n"
        f"- **Time:** {result.processing_time_ms:.0f} ms\n\n"
        f"**Reasoning:**\n\n{f.reasoning}"
    ))

<>:38: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.
<>:38: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.
/var/folders/tl/xzkhv7fn50l5d538d_n58wl80000gn/T/ipykernel_24828/1751952354.py:38: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.
  f"- **Cost:** \${result.total_cost_usd:.6f}\n"


## Configuration

Edit `QUESTION` and `PROVIDERS` below. `PROVIDERS` needs credentials in `.env` for each provider you list (see `.env.example`). Two or three cheap providers is enough for a quick demo.

In [2]:
QUESTION = "Will SpaceX land humans on Mars by end of 2030?"
PROVIDERS = ["anthropic", "google"]  # add more, e.g. "moonshot", "openai", "deepseek", "alibaba", "meta"
SINGLE_PROVIDER = "anthropic"        # provider used for the single-LLM baseline

## 1. Single LLM (baseline)

One model, one call. Same as asking Claude or ChatGPT directly.

In [3]:
single = SingleLLM(Agent(SINGLE_PROVIDER, DEFAULT_MODELS[SINGLE_PROVIDER]))
single_result = await single.run(QUESTION)
show(single_result, title=f"Single LLM ({SINGLE_PROVIDER}/{DEFAULT_MODELS[SINGLE_PROVIDER]})")

### Single LLM (anthropic/claude-haiku-4-5)
- **Prediction:** 0.08
- **Confidence:** 0.72
- **Cost:** \$0.000835
- **Tokens:** 135 in / 140 out
- **Time:** 2599 ms

**Reasoning:**

SpaceX has ambitious Mars goals and is developing Starship, but landing humans on Mars by end of 2030 requires completing multiple unprecedented milestones: Starship orbital refueling, moon missions, Mars trajectory capability, entry/descent/landing systems, and human-rated vehicles—all within ~6 years. Historical spaceflight timelines and technical complexity suggest this is extremely unlikely, though SpaceX's track record of exceeding expectations provides some non-negligible probability.

## 2. Voting Swarm

Each provider answers independently in parallel; the final prediction is a uniform weighted average. Cost and tokens sum across all calls.

In [4]:
swarm = VotingSwarm(make_agents(PROVIDERS))
swarm_result = await swarm.run(QUESTION)
show(swarm_result, title=f"Voting Swarm ({', '.join(PROVIDERS)})")

### Voting Swarm (anthropic, google)
- **Prediction:** 0.08
- **Confidence:** 0.80
- **Cost:** \$0.001202
- **Tokens:** 258 in / 283 out
- **Time:** 6220 ms

**Reasoning:**

[claude-haiku-4-5 | weight=0.50 | p=0.08] SpaceX aims for Mars missions but faces enormous technical and logistical hurdles. A crewed landing by end of 2030 requires completing Starship development, multiple orbital refueling tests, and a multi-month journey with life support systems—all within ~6 years. While SpaceX moves faster than traditional aerospace, this timeline is extremely compressed compared to expert consensus estimates of 2035-2040 or later.

[gemini-2.5-flash | weight=0.50 | p=0.08] While SpaceX is making rapid progress with Starship, landing humans on Mars by the end of 2030 presents immense technical and logistical challenges. Before a crewed mission, multiple uncrewed Starships would need to successfully reach Mars, demonstrate precise entry, descent, and landing (EDL), and likely establish initial infrastructure. Human-rating Starship, developing robust life support, radiation shielding, and proving orbital refueling reliability for such a long-duration mission adds several more years of rigorous testing and development beyond current milestones, making the 2030 timeline highly improbable.

## 3. LLM Council

Same agents as the swarm, but with an extra round: each agent scores the *reasoning quality* of anonymized peer forecasts on a 1–10 scale. Final weights come from the mean score each agent received from peers (self-scores dropped). If all critics fail parsing, weights fall back to uniform (i.e. degrades to a swarm).

Expect ~2x the tokens/cost of a same-size swarm — one call per agent for the forecast, one for the critique.

In [5]:
council = LLMCouncil(make_agents(PROVIDERS))
council_result = await council.run(QUESTION)
show(council_result, title=f"LLM Council ({', '.join(PROVIDERS)})")

### LLM Council (anthropic, google)
- **Prediction:** 0.08
- **Confidence:** 0.79
- **Cost:** \$0.001841
- **Tokens:** 1107 in / 291 out
- **Time:** 17231 ms

**Reasoning:**

[claude-haiku-4-5 | peer-weight=0.56 | p=0.08] SpaceX has made ambitious statements about Mars landings, but landing humans on Mars by end of 2030 requires completing: Starship orbital refueling tests, multiple successful uncrewed Mars missions, life support systems validation, and crewed lunar missions first. Current timelines suggest 2032-2035 are more realistic for early human Mars landings. Only ~8 years remain, and technical challenges typically exceed optimistic projections.

[gemini-2.5-flash | peer-weight=0.44 | p=0.07] SpaceX faces immense technical and logistical hurdles, including achieving reliable Starship orbital flights, in-orbit refueling, uncrewed Mars landings, and human-rating the vehicle for deep space. The aggressive timeline (less than 7 years) for these unprecedented developments, coupled with the need for robust life support, radiation shielding, and return capability, makes a human landing by end of 2030 highly improbable, despite SpaceX's rapid pace.

## Comparison

In [6]:
rows = [
    ("Single LLM", single_result),
    ("Voting Swarm", swarm_result),
    ("LLM Council", council_result),
]

html = [
    "<table>",
    "<tr><th>Config</th><th>Prediction</th><th>Confidence</th>"
    "<th>Input tok</th><th>Output tok</th><th>Cost (USD)</th><th>Time (ms)</th></tr>",
]
for label, r in rows:
    html.append(
        f"<tr><td>{label}</td>"
        f"<td>{r.forecast.prediction:.2f}</td>"
        f"<td>{r.forecast.confidence:.2f}</td>"
        f"<td>{r.total_input_tokens}</td>"
        f"<td>{r.total_output_tokens}</td>"
        f"<td>${r.total_cost_usd:.6f}</td>"
        f"<td>{r.processing_time_ms:.0f}</td></tr>"
    )
html.append("</table>")
HTML("".join(html))

Config,Prediction,Confidence,Input tok,Output tok,Cost (USD),Time (ms)
Single LLM,0.08,0.72,135,140,$0.000835,2599
Voting Swarm,0.08,0.80,258,283,$0.001202,6220
LLM Council,0.08,0.79,1107,291,$0.001841,17231
